In [8]:
import pandas as pd
df=pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,neutral
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


In [9]:
def email_assistant(email_text):
    text = email_text.lower()

    if "urgent" in text or "deadline" in text:
        return "notify", "urgent"
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"
    else:
        return "respond", "neutral"


In [10]:
#define Dangerous Actions
dangerous_actions = ["respond"]


In [11]:
#human-in-the-loop check
def hitl_check(action):

    if action in dangerous_actions:
       return "Wait_for_Human"
    return "Auto_Approve"


In [12]:
# Simulate Human approval
def human_approval():
    decision = input("Approve action? (yes/no): ")
    return decision.lower() == "yes"
   

In [14]:
results = []

for _, row in df.sample(5).iterrows():
    action, tone = email_assistant(row["body"])

    status = hitl_check(action)

    if status == "Wait_for_Human":
        approved = human_approval()
        final_action = action if approved else "blocked"
    else:
        final_action = action

    results.append({
        "email_snippet": row["body"][:50],  # first 50 chars
        "ai_action": action,
        "final_action": final_action,
        "hitl_status": status
    })

eval_df = pd.DataFrame(results)
print(eval_df)



                                       email_snippet ai_action final_action  \
0  Notice: Your account will be locked unless ver...   respond      respond   
1  Your order #9129 has been shipped and is expec...   respond      blocked   
2  Please complete the mandatory training module ...    notify       notify   
3  Please complete the mandatory training module ...    notify       notify   
4  Security alert: multiple failed login attempts...   respond      blocked   

      hitl_status  
0  Wait_for_Human  
1  Wait_for_Human  
2    Auto_Approve  
3    Auto_Approve  
4  Wait_for_Human  


In [15]:
eval_df.to_csv("../data/milestone3_HITL_output.csv", index=False)
